# RFM Customer Segmentation - Clustering
### Based on: Wong, Tong & Haw (2024) - Exploring Customer Segmentation in E-Commerce using RFM Analysis with Clustering Techniques
### Journal: Journal of Telecommunications and the Digital Economy, Vol. 12 No. 3, pp. 97–125
### DOI: https://doi.org/10.18080/jtde.v12n3.978

In [0]:
%restart_python

In [0]:
import sys
import io

_old_stderr = sys.stderr
sys.stderr = io.StringIO()
import threadpoolctl
threadpoolctl.threadpool_limits(1)
sys.stderr = _old_stderr

import warnings
warnings.filterwarnings("ignore")

import os
os.environ["PYTHONWARNINGS"] = "ignore"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import PowerTransformer
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score

import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

from pyspark.sql.functions import col, count, when, avg, stddev
from pyspark.sql.functions import min as spark_min, max as spark_max

COLOR_MAIN    = "#1B3A6B"
COLOR_WARN    = "#E65C00"
COLOR_OK      = "#00695C"
COLOR_BAD     = "#B71C1C"
COLOR_NEUTRAL = "#9E9E9E"

print("imports done")

**Load RFM Features**

In [0]:
churn_features = spark.table("bharatmart.ml.churn_features")

rfm_df = churn_features.select("customer_id", "recency_days", "total_orders", "total_spent")

print(f"rows : {rfm_df.count():,}")

In [0]:
display(rfm_df.limit(5))

**Clip total_spent at p99**

In [0]:
rfm_pdf = rfm_df.toPandas()
rfm_pdf["total_orders"] = rfm_pdf["total_orders"].astype(float)
rfm_pdf["total_spent"]  = rfm_pdf["total_spent"].astype(float)

# compute p99 fresh each run — not hardcoded
# as new customers are added the threshold adjusts automatically
p99_spent = float(np.percentile(rfm_pdf["total_spent"], 99))
print(f"p99 total_spent today = Rs. {p99_spent:,.0f}")

rfm_pdf["total_spent"] = rfm_pdf["total_spent"].clip(upper=p99_spent)

print(f"max total_spent after clip = Rs. {rfm_pdf['total_spent'].max():,.0f}")
print(f"rows : {len(rfm_pdf):,}")

p99 is computed fresh from today's data - not hardcoded.

every night when the batch runs, this number recalculates from 
whoever is in churn_features at that point. if the customer base 
grows and a genuine high-spender joins at Rs. 350,000 next month, 
the p99 threshold moves up to reflect that. their monetary value 
stays accurate and they land in the right segment.

hardcoding Rs. 307,047 would silently break this over time.

rows still 92,107 - no one dropped, only the top 1% of spenders 
got their value capped.

**Normalise - Power Transformer Yeo-Johnson**

In [0]:
rfm_features = rfm_pdf[["recency_days", "total_orders", "total_spent"]].copy()

scaler = PowerTransformer(method="yeo-johnson")
rfm_scaled = scaler.fit_transform(rfm_features)

rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=["recency_scaled", "orders_scaled", "spent_scaled"])

print("yeo-johnson applied")
print(f"shape : {rfm_scaled_df.shape}")
print()
print(rfm_scaled_df.describe().round(3))

yeo-johnson worked. all three features now have mean = 0 and std = 1.

before normalisation the means were completely different scales - 
recency mean 81 days, orders mean 17, spent mean 20,548. K-Means 
would have clustered almost entirely on spent because that dimension 
dominated the euclidean distance calculation.

now all three sit on the same scale. each feature gets an equal 
vote in where the cluster boundaries land.

the ranges are reasonable too. recency goes -1.65 to 3.75, orders 
-2.16 to 2.83, spent -4.56 to 2.14. no feature is wildly wider 
than the others. spent still has a slightly wider range on the 
negative side, that's the large number of low-spending customers 
we saw in EDA pulling the left tail out.

Wong et al. (2024) showed the same result after Yeo-Johnson 
the distribution became more symmetrical compared to log 
transformation. our std = 1.0 across all three confirms the 
normalisation is clean.

**Distribution Before vs After Normalisation**

In [0]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# before
axes[0][0].hist(rfm_pdf["recency_days"], bins=50, color=COLOR_BAD, edgecolor="white")
axes[0][0].set_title("Recency — Before")
axes[0][0].set_xlabel("recency_days")

axes[0][1].hist(rfm_pdf["total_orders"], bins=50, color=COLOR_BAD, edgecolor="white")
axes[0][1].set_title("Frequency — Before")
axes[0][1].set_xlabel("total_orders")

axes[0][2].hist(rfm_pdf["total_spent"], bins=50, color=COLOR_BAD, edgecolor="white")
axes[0][2].set_title("Monetary — Before")
axes[0][2].set_xlabel("total_spent")

# after
axes[1][0].hist(rfm_scaled_df["recency_scaled"], bins=50, color=COLOR_OK, edgecolor="white")
axes[1][0].set_title("Recency — After Yeo-Johnson")
axes[1][0].set_xlabel("recency_scaled")

axes[1][1].hist(rfm_scaled_df["orders_scaled"], bins=50, color=COLOR_OK, edgecolor="white")
axes[1][1].set_title("Frequency — After Yeo-Johnson")
axes[1][1].set_xlabel("orders_scaled")

axes[1][2].hist(rfm_scaled_df["spent_scaled"], bins=50, color=COLOR_OK, edgecolor="white")
axes[1][2].set_title("Monetary — After Yeo-Johnson")
axes[1][2].set_xlabel("spent_scaled")

plt.suptitle("RFM Distributions — Before vs After Yeo-Johnson Normalisation", y=1.01)
plt.tight_layout()
plt.show()

the top row vs bottom row tells the whole story.

recency before - one massive bar on the left, everything else flat. 
recency after - spread across -1.5 to 3.7. still not perfectly 
symmetric but the extreme compression is gone. 
K-Means can now actually differentiate between customers at different recency levels.

frequency before - readable but with that sharp spike at 19 orders. 
frequency after - the spike is still there around orders_scaled = 1.0 but it no longer dominates. 
the distribution is more spread out.

monetary before - 80,000 customers in one bar, unreadable. 
monetary after - the mass has spread from -4.5 to 2.1. the bulk of customers sit between -1 and 0.5 which matches what we found 
in EDA. 
most customers spent under Rs. 2,000 lifetime.

one thing to note on recency_scaled - there are two visible humps. 
one around -0.7 to 0 and another around 0.7 to 1. this is the 
active vs churned customer split showing up naturally in the data. 
Yeo-Johnson did not hide this structure, it preserved it. 
that separation is exactly what clustering needs to find segments.

Wong et al. (2024) showed the same improvement in their Figure 5 - 
Power Transformer produced a more symmetrical distribution compared 
to log transformation shown in Figure 4. our bottom row matches 
their result.

**Elbow Method - Find Optimal K**

In [0]:
mlflow.autolog(disable=True)

inertias = []
k_range  = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled_df)
    inertias.append(km.inertia_)
    print(f"k={k}  inertia={km.inertia_:,.1f}")

**Elbow Method Plot**

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(list(k_range), inertias, marker="o", color=COLOR_MAIN, linewidth=2)

for k, inertia in zip(k_range, inertias):
    ax.annotate(f"{inertia:,.0f}", (k, inertia),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8)

ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Inertia (WCSS)")
ax.set_title("Elbow Method — Optimal K for K-Means")
ax.set_xticks(list(k_range))
plt.tight_layout()
plt.show()

the elbow is clearly at K=3.

from k=2 to k=3 inertia drops 65,742 - the steepest drop on the 
entire curve. from k=3 onwards the curve flattens out. each 
additional cluster after 3 gives less and less improvement.

this matches Wong et al. (2024) who found the same elbow at K=3 
on their e-commerce dataset.

but elbow method alone is not enough - it is visual and subjective. 
someone could argue k=4 looks reasonable too. we validate with 
Silhouette Score and Calinski-Harabasz next to confirm objectively.

**Silhouette Score - K=2 to 10**

In [0]:
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled_df)
    score  = silhouette_score(rfm_scaled_df, labels)
    sil_scores.append(score)
    print(f"k={k}  silhouette={score:.4f}")

K=3 wins clearly - silhouette score 0.4108 is the highest across 
all values from 2 to 10.

every other K sits between 0.33 and 0.35. K=3 jumps to 0.41 - 
a meaningful gap, not a marginal difference.

this confirms the elbow plot was right. K=3 is not just where 
the inertia curve bends - it also produces the most well-separated 
clusters on BharatMart data.

Wong et al. (2024) reported silhouette score of 0.47 for 
Hierarchical Clustering at K=3 on their dataset. our K-Means 
at K=3 gives 0.41. slightly lower which is expected - we are 
running K-Means here, not Hierarchical. the paper found 
Hierarchical outperformed K-Means on silhouette score. 
we will see if the same holds on BharatMart data when we run 
Hierarchical next.

two out of three validation methods now point to K=3. 
Calinski-Harabasz next.

**Calinski-Harabasz Index - K=2 to 10**

In [0]:
ch_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled_df)
    score = calinski_harabasz_score(rfm_scaled_df, labels)
    ch_scores.append(score)
    print(f"k={k}  calinski_harabasz={score:.1f}")

K=3 wins again - 71,482.9 is the highest Calinski-Harabasz score 
across all K values tested.

after K=3 the scores drop to the 63,000-67,000 range and stay 
there. K=3 is clearly ahead.

all three methods now agree - elbow, silhouette, and 
Calinski-Harabasz all point to K=3.

Wong et al. (2024) reported Calinski-Harabasz of 3,787.1 for 
Hierarchical Clustering at K=3. our K-Means number is 71,482.9 - 
much higher. this is expected. their dataset had 3,700 customers, 
ours has 92,107. Calinski-Harabasz scales with dataset size so 
direct comparison of the absolute number is not meaningful. 
what matters is that K=3 is the winner within our own dataset, 
which it clearly is.

K=3 is confirmed. locking it in.

**Validation Summary Plot**

In [0]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(list(k_range), inertias, marker="o", color=COLOR_MAIN, linewidth=2)
axes[0].set_title("Elbow Method (WCSS)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")
axes[0].set_xticks(list(k_range))
axes[0].axvline(x=3, color=COLOR_BAD, linestyle="--", linewidth=1.5, label="K=3")
axes[0].legend()

axes[1].plot(list(k_range), sil_scores, marker="o", color=COLOR_OK, linewidth=2)
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Score")
axes[1].set_xticks(list(k_range))
axes[1].axvline(x=3, color=COLOR_BAD, linestyle="--", linewidth=1.5, label="K=3")
axes[1].legend()

axes[2].plot(list(k_range), ch_scores, marker="o", color=COLOR_WARN, linewidth=2)
axes[2].set_title("Calinski-Harabasz Index")
axes[2].set_xlabel("K")
axes[2].set_ylabel("Score")
axes[2].set_xticks(list(k_range))
axes[2].axvline(x=3, color=COLOR_BAD, linestyle="--", linewidth=1.5, label="K=3")
axes[2].legend()

plt.suptitle("K Validation — All Three Methods Agree on K=3", y=1.02)
plt.tight_layout()
plt.show()

three charts, one answer - K=3.

elbow plot: the bend is at K=3. after that the curve flattens.

silhouette: K=3 is the clear peak at 0.41. every other K sits 
below 0.36. the drop after K=3 is sharp - adding a 4th cluster 
actually makes the segments worse, not better.

calinski-harabasz: K=3 peaks at 71,482. drops to 65,000 range 
after that and stays flat.

all three independent methods land on the same number. 
K=3 is confirmed for BharatMart data.

this also independently validates Wong et al. (2024) who found 
K=3 optimal on a UK e-commerce dataset. we are now seeing the 
same result on an Indian e-commerce dataset with 25x more 
customers. the finding holds across different markets.

K=3 is locked. moving to fit K-Means and Hierarchical.